# Mistral-7B Model Loading Test

This notebook tests the PyTorch → JAX conversion pipeline for Mistral-7B.

**What this notebook does:**
1. Loads Mistral-7B from HuggingFace (~14GB download)
2. Converts PyTorch weights to JAX
3. Inspects weight structure to verify conversion
4. Validates architecture matches expectations

**Requirements:**
- Colab with sufficient RAM (free tier should work)
- ~14GB download (fast on Colab)
- No GPU needed for this test

## 1. Setup Environment

Install required packages (if not already installed).

In [1]:
# Check if we're in Colab
try:
    import google.colab
    IN_COLAB = True
    print("✓ Running in Google Colab")
except:
    IN_COLAB = False
    print("✓ Running locally")

# Install dependencies (uncomment if needed)
# !pip install -q torch transformers jax jaxlib flax

✓ Running in Google Colab


## 2. Load Project Code

**Option A: Clone from GitHub** (if you've pushed your code)
```python
!git clone https://github.com/yourusername/LLM_Response_Time_Optimizer.git
%cd LLM_Response_Time_Optimizer
```

**Option B: Upload files manually** (if testing locally first)
- Upload the entire `src/` folder
- Make sure `model_conversion.py` and `cached_generation.py` are present

In [2]:
!git clone https://github.com/YashM246/LLM_Response_Time_Optimizer.git

Cloning into 'LLM_Response_Time_Optimizer'...
remote: Enumerating objects: 314, done.
remote: Counting objects: 100% (54/54), done.
remote: Compressing objects: 100% (39/39), done.
remote: Total 314 (delta 28), reused 37 (delta 15), pack-reused 260 (from 1)
Receiving objects: 100% (314/314), 192.49 KiB | 3.50 MiB/s, done.
Resolving deltas: 100% (181/181), done.


In [3]:
# If using Option B (manual upload), uncomment and run:
# from google.colab import files
# print("Please upload the 'src' folder as a zip file")
# uploaded = files.upload()
# !unzip -q src.zip

import sys
sys.path.append('./LLM_Response_Time_Optimizer/')

# Verify imports work
try:
    from src.model_conversion import convert_model
    from src.cached_generation import MISTRAL_CONFIG, get_model
    print("✓ Successfully imported project modules")
except ImportError as e:
    print(f"✗ Import failed: {e}")
    print("Please make sure src/ folder is in the current directory")

✓ Successfully imported project modules


## 3. Load Mistral-7B Model

This will:
- Download Mistral-7B weights (~14GB, takes 2-5 minutes on Colab)
- Convert PyTorch → JAX
- Build PyTree structure

In [4]:
import time

print("=" * 70)
print("Starting Mistral-7B Loading Test")
print("=" * 70)
print("\nThis will download ~14GB of model weights.")
print("Expected time: 2-5 minutes on Colab\n")

start_time = time.time()

# Load and convert model
params, tokenizer, model_type = convert_model(model_type="mistral")

elapsed = time.time() - start_time
print(f"\n{'='*70}")
print(f"✓ Conversion completed in {elapsed:.1f} seconds ({elapsed/60:.1f} minutes)")
print(f"{'='*70}")

Starting Mistral-7B Loading Test

This will download ~14GB of model weights.
Expected time: 2-5 minutes on Colab


PyTorch -> JAX Conversion Pipeline (MISTRAL)

[1/4] Loading PyTorch model...
Loading Mistral-7B model...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/996 [00:00<?, ?B/s]

tokenizer.model:   0%|          | 0.00/493k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

...

model-00002-of-00002.safetensors:   0%|          | 0.00/4.54G [00:00<?, ?B/s]

model-00001-of-00002.safetensors:   0%|          | 0.00/9.94G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

[OK] Loaded 291 parameters from mistralai/Mistral-7B-v0.1
Example weight keys:
    model.embed_tokens.weight: torch.Size([32000, 4096])
    model.layers.0.self_attn.q_proj.weight: torch.Size([4096, 4096])
    model.layers.0.self_attn.k_proj.weight: torch.Size([1024, 4096])
    model.layers.0.self_attn.v_proj.weight: torch.Size([1024, 4096])
    model.layers.0.self_attn.o_proj.weight: torch.Size([4096, 4096])

[2/4] Converting to JAX arrays...
  [OK] Transposed model.layers.0.self_attn.q_proj.weight: torch.Size([4096, 4096]) -> (4096, 4096)
  [OK] Transposed model.layers.0.self_attn.k_proj.weight: torch.Size([1024, 4096]) -> (4096, 1024)
  [OK] Transposed model.layers.0.self_attn.v_proj.weight: torch.Size([1024, 4096]) -> (4096, 1024)
  [OK] Transposed model.layers.0.self_attn.o_proj.weight: torch.Size([4096, 4096]) -> (4096, 4096)
  [OK] Transposed model.layers.0.mlp.gate_proj.weight: torch.Size([14336, 4096]) -> (4096, 14336)
  [OK] Transposed model.layers.0.mlp.up_proj.weight: torch.

## 4. Verify Conversion Success

Let's inspect the converted weights to ensure they match Mistral's architecture.

In [5]:
print("=" * 70)
print("Conversion Verification")
print("=" * 70)

# 1. Basic info
print(f"\n[1] Model Type: {model_type}")
print(f"[2] Tokenizer Vocab Size: {len(tokenizer)}")
print(f"[3] Top-level keys: {list(params.keys())}")

# 2. Inspect structure
if 'params' in params:
    model_keys = list(params['params'].keys())
    print(f"\n[4] Model keys: {model_keys}")

    # Check for Mistral-specific structure
    if 'model' in params['params']:
        print("\n✓ Found 'model' key (Mistral structure detected)")

        model_params = params['params']['model']
        model_subkeys = list(model_params.keys())
        print(f"   Model subkeys: {model_subkeys}")

        # Check layers
        if 'layers' in model_params:
            num_layers = len(model_params['layers'])
            print(f"\n✓ Number of layers: {num_layers}")
            print(f"   Expected: 32 layers (Mistral-7B)")

            if num_layers == 32:
                print("   ✓ Layer count matches Mistral-7B!")
            else:
                print(f"   ✗ WARNING: Expected 32 layers, got {num_layers}")

        # Check embeddings
        if 'embed_tokens' in model_params:
            embed_shape = model_params['embed_tokens']['embedding'].shape
            print(f"\n✓ Token embeddings shape: {embed_shape}")
            print(f"   Expected: (32000, 4096) [vocab_size, hidden_dim]")

            if embed_shape == (32000, 4096):
                print("   ✓ Embedding shape matches Mistral-7B!")

        # Check final norm
        if 'norm' in model_params:
            norm_keys = list(model_params['norm'].keys())
            print(f"\n✓ Final norm keys: {norm_keys}")
            if 'kernel' in norm_keys:
                norm_shape = model_params['norm']['kernel'].shape
                print(f"   Norm weight shape: {norm_shape}")
                print(f"   Expected: (4096,) [hidden_dim]")

    # Check LM head
    if 'lm_head' in params['params']:
        lm_head_keys = list(params['params']['lm_head'].keys())
        print(f"\n✓ LM head keys: {lm_head_keys}")
        if 'kernel' in lm_head_keys:
            lm_shape = params['params']['lm_head']['kernel'].shape
            print(f"   LM head shape: {lm_shape}")
            print(f"   Expected: (4096, 32000) [hidden_dim, vocab_size]")

Conversion Verification

[1] Model Type: mistral
[2] Tokenizer Vocab Size: 32000
[3] Top-level keys: ['params']

[4] Model keys: ['model', 'lm_head']

✓ Found 'model' key (Mistral structure detected)
   Model subkeys: ['embed_tokens', 'layers', 'norm']

✓ Number of layers: 32
   Expected: 32 layers (Mistral-7B)
   ✓ Layer count matches Mistral-7B!

✓ Token embeddings shape: (32000, 4096)
   Expected: (32000, 4096) [vocab_size, hidden_dim]
   ✓ Embedding shape matches Mistral-7B!

✓ Final norm keys: ['kernel']
   Norm weight shape: (4096,)
   Expected: (4096,) [hidden_dim]

✓ LM head keys: ['kernel']
   LM head shape: (4096, 32000)
   Expected: (4096, 32000) [hidden_dim, vocab_size]


## 5. Inspect Layer Structure

Let's examine the first transformer layer to verify all Mistral-specific components are present.

In [6]:
print("=" * 70)
print("Layer 0 Structure Inspection")
print("=" * 70)

if 'model' in params['params'] and 'layers' in params['params']['model']:
    layer_0 = params['params']['model']['layers']['0']
    layer_keys = list(layer_0.keys())

    print(f"\nLayer 0 top-level keys: {layer_keys}")
    print(f"\nExpected Mistral components:")
    print("  - input_layernorm (RMSNorm before attention)")
    print("  - self_attn (with q_proj, k_proj, v_proj, o_proj)")
    print("  - post_attention_layernorm (RMSNorm before MLP)")
    print("  - mlp (with gate_proj, up_proj, down_proj for SwiGLU)")

    # Check each component
    print(f"\n{'─'*70}")
    print("Component Verification:")
    print(f"{'─'*70}")

    # 1. Input LayerNorm
    if 'input_layernorm' in layer_keys:
        ln_keys = list(layer_0['input_layernorm'].keys())
        print(f"\n✓ input_layernorm keys: {ln_keys}")
        if 'kernel' in ln_keys:
            shape = layer_0['input_layernorm']['kernel'].shape
            print(f"   Shape: {shape} (expected: (4096,))")
    else:
        print("\n✗ Missing input_layernorm")

    # 2. Self Attention
    if 'self_attn' in layer_keys:
        attn_keys = list(layer_0['self_attn'].keys())
        print(f"\n✓ self_attn keys: {attn_keys}")

        expected_attn_keys = ['q_proj', 'k_proj', 'v_proj', 'o_proj']
        for key in expected_attn_keys:
            if key in attn_keys:
                proj_keys = list(layer_0['self_attn'][key].keys())
                shape = layer_0['self_attn'][key]['kernel'].shape if 'kernel' in proj_keys else None
                print(f"   ✓ {key}: {proj_keys}, shape: {shape}")
            else:
                print(f"   ✗ Missing {key}")
    else:
        print("\n✗ Missing self_attn")

    # 3. Post-Attention LayerNorm
    if 'post_attention_layernorm' in layer_keys:
        ln_keys = list(layer_0['post_attention_layernorm'].keys())
        print(f"\n✓ post_attention_layernorm keys: {ln_keys}")
        if 'kernel' in ln_keys:
            shape = layer_0['post_attention_layernorm']['kernel'].shape
            print(f"   Shape: {shape} (expected: (4096,))")
    else:
        print("\n✗ Missing post_attention_layernorm")

    # 4. MLP
    if 'mlp' in layer_keys:
        mlp_keys = list(layer_0['mlp'].keys())
        print(f"\n✓ mlp keys: {mlp_keys}")

        expected_mlp_keys = ['gate_proj', 'up_proj', 'down_proj']
        for key in expected_mlp_keys:
            if key in mlp_keys:
                proj_keys = list(layer_0['mlp'][key].keys())
                shape = layer_0['mlp'][key]['kernel'].shape if 'kernel' in proj_keys else None
                print(f"   ✓ {key}: {proj_keys}, shape: {shape}")
            else:
                print(f"   ✗ Missing {key}")
    else:
        print("\n✗ Missing mlp")
else:
    print("\n✗ Could not access layer structure")

Layer 0 Structure Inspection

Layer 0 top-level keys: ['self_attn', 'mlp', 'input_layernorm', 'post_attention_layernorm']

Expected Mistral components:
  - input_layernorm (RMSNorm before attention)
  - self_attn (with q_proj, k_proj, v_proj, o_proj)
  - post_attention_layernorm (RMSNorm before MLP)
  - mlp (with gate_proj, up_proj, down_proj for SwiGLU)

──────────────────────────────────────────────────────────────────────
Component Verification:
──────────────────────────────────────────────────────────────────────

✓ input_layernorm keys: ['kernel']
   Shape: (4096,) (expected: (4096,))

✓ self_attn keys: ['q_proj', 'k_proj', 'v_proj', 'o_proj']
   ✓ q_proj: ['kernel'], shape: (4096, 4096)
   ✓ k_proj: ['kernel'], shape: (4096, 1024)
   ✓ v_proj: ['kernel'], shape: (4096, 1024)
   ✓ o_proj: ['kernel'], shape: (4096, 4096)

✓ post_attention_layernorm keys: ['kernel']
   Shape: (4096,) (expected: (4096,))

✓ mlp keys: ['gate_proj', 'up_proj', 'down_proj']
   ✓ gate_proj: ['kernel'], 

## 6. Verify Configuration Match

Compare expected Mistral configuration with actual converted weights.

In [7]:
print("=" * 70)
print("Configuration Validation")
print("=" * 70)

# Get expected config
expected_config = get_model("mistral")

print("\nExpected Mistral-7B Configuration:")
print(f"{'─'*70}")
for key, value in expected_config.items():
    print(f"  {key:20s}: {value}")

print(f"\n{'─'*70}")
print("Validation Checks:")
print(f"{'─'*70}")

checks = []

# Check vocab size
if len(tokenizer) == expected_config['vocab_size']:
    checks.append(("✓", "Vocab size", f"{len(tokenizer)} == {expected_config['vocab_size']}"))
else:
    checks.append(("✗", "Vocab size", f"{len(tokenizer)} != {expected_config['vocab_size']}"))

# Check number of layers
if 'model' in params['params'] and 'layers' in params['params']['model']:
    num_layers = len(params['params']['model']['layers'])
    if num_layers == expected_config['num_layers']:
        checks.append(("✓", "Number of layers", f"{num_layers} == {expected_config['num_layers']}"))
    else:
        checks.append(("✗", "Number of layers", f"{num_layers} != {expected_config['num_layers']}"))

# Check embedding dimension
if 'model' in params['params'] and 'embed_tokens' in params['params']['model']:
    embed_dim = params['params']['model']['embed_tokens']['embedding'].shape[1]
    if embed_dim == expected_config['hidden_dim']:
        checks.append(("✓", "Hidden dimension", f"{embed_dim} == {expected_config['hidden_dim']}"))
    else:
        checks.append(("✗", "Hidden dimension", f"{embed_dim} != {expected_config['hidden_dim']}"))

# Print results
for status, check_name, result in checks:
    print(f"{status} {check_name:25s}: {result}")

# Final summary
passed = sum(1 for s, _, _ in checks if s == "✓")
total = len(checks)

print(f"\n{'='*70}")
if passed == total:
    print(f"✓ ALL CHECKS PASSED ({passed}/{total})")
    print("✓ Mistral-7B conversion successful!")
else:
    print(f"⚠ SOME CHECKS FAILED ({passed}/{total} passed)")
    print("⚠ Please review the discrepancies above")
print(f"{'='*70}")

Configuration Validation

Expected Mistral-7B Configuration:
──────────────────────────────────────────────────────────────────────
  num_layers          : 32
  num_heads           : 32
  num_kv_heads        : 8
  hidden_dim          : 4096
  head_dim            : 128
  intermediate_dim    : 14336
  vocab_size          : 32000
  max_seq_len         : 32768
  use_rope            : True
  use_rms_norm        : True
  use_swiglu          : True
  rope_theta          : 10000.0

──────────────────────────────────────────────────────────────────────
Validation Checks:
──────────────────────────────────────────────────────────────────────
✓ Vocab size               : 32000 == 32000
✓ Number of layers         : 32 == 32
✓ Hidden dimension         : 4096 == 4096

✓ ALL CHECKS PASSED (3/3)
✓ Mistral-7B conversion successful!


## 7. Summary

If all checks passed:
- ✅ Mistral-7B weights successfully loaded and converted to JAX
- ✅ PyTree structure matches expected Mistral architecture
- ✅ All components present (RMSNorm, separate Q/K/V, SwiGLU)
- ✅ Ready for text generation testing

**Next steps:**
1. Test text generation with Mistral (will need GPU)
2. Compare generation quality with GPT-2
3. Benchmark performance improvements

In [8]:
# Save this info for future reference
print("\nConversion Summary:")
print(f"  Model: {model_type}")
print(f"  Parameters: {len(params['params'])} top-level groups")
print(f"  Tokenizer vocab: {len(tokenizer)}")
print(f"  Ready for generation: Yes (with GPU recommended)")


Conversion Summary:
  Model: mistral
  Parameters: 2 top-level groups
  Tokenizer vocab: 32000
  Ready for generation: Yes (with GPU recommended)
